In [11]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks
# Obtener la ruta del directorio raíz del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))
# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(PROJECT_ROOT)

INPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'inputs')
OUTPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'outputs')

def get_file_from_path(file_path: str) -> str:
    return os.path.join(INPUTS_FOLDER, file_path)

test_files = [
    'banorte_credit_new.pdf',
    'banorte_credit_old.pdf',
    'banorte_debit.pdf',
    'bbva_debit.pdf',
]

file_path = get_file_from_path('banorte_credit_old.pdf')

In [12]:
from models import DocumentProcessingFacade
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

doc_processor = DocumentProcessingFacade(file_path)
statement_properties = doc_processor.get_statement_properties() 
bank = statement_properties['bank']
statement_type = statement_properties['statement_type']
new_format = statement_properties['new_format']

print(bank, ' - ', statement_type, ' - ', new_format)

banorte  -  credit  -  False


In [13]:
#extracted_words = doc_processor.get_extracted_words()
#
#if statement_type == 'debit':
#    extracted_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_extracted_words.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    extracted_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_extracted_words.csv')
#    
#extracted_words.to_csv(extracted_words_path, index=False)

In [14]:
corrected_extracted_words = doc_processor.get_corrected_extracted_words()
corrected_extracted_words

,page,text,x0,top,x1,bottom
0,1,Estado,466.251000,11.71425,495.615000,23.71425
1,1,Clásica,504.141000,18.91425,533.865000,30.91425
2,1,de,451.455000,23.86425,462.483000,35.86425
3,1,Cuenta,464.691000,23.86425,495.615000,35.86425
4,1,MARIA,30.440250,94.58445,54.989636,104.58420
...,...,...,...,...,...,...
2444,6,Monterrey,295.919093,770.44620,333.858144,780.44595
2445,6,Nuevo,335.698098,770.44620,359.217510,780.44595
2446,6,"León,",361.057464,770.44620,380.886968,780.44595
2447,6,C.P.,382.726922,770.44620,394.866619,780.44595


In [15]:
#if statement_type == 'debit':
#    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_corrected_words.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_corrected_words.csv')
#    
#corrected_extracted_words.to_csv(corrected_words_path, index=False)

In [16]:
from models import TableProcessingFacade

table_processor = TableProcessingFacade(corrected_extracted_words, statement_properties)
reconstructor = table_processor.get_reconstructor()
structured_table = reconstructor.get_structured_table()
structured_table

,Date,Description,Importe
0,None,Fecha Concepto RFC/CURP Tipo de transacción Im...,None
1,11/12,y d GASOL CHICHI 2 MERIDA YUC MX CYU 941027P30,$700.00
2,15/12,y a CHEDRAUI MERIDA VI MERIDA YUC MX TCH 85070...,$151.10
3,16/12,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,$153.46
4,16/12,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,$86.78
5,21/12,"i SU PAGO, GRACIAS","$1,865.00-"
6,None,1/6,None
7,None,"Banco Mercantil del Norte, S.A., Institución d...",None
8,None,"Av. Revolución 3000, Colonia La Primavera, Mon...",None
9,None,Estado,None


In [17]:
reconstructed_table = table_processor.reconstruct_table()
reconstructed_table

,Date,Description,Importe
0,11/12,y d GASOL CHICHI 2 MERIDA YUC MX CYU 941027P30,$700.00
1,15/12,y a CHEDRAUI MERIDA VI MERIDA YUC MX TCH 85070...,$151.10
2,16/12,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,$153.46
3,16/12,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,$86.78
4,21/12,"i SU PAGO, GRACIAS 1/6 Banco Mercantil del ...","$1,865.00-"
5,23/12,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,$79.00
6,24/12,y a GM 8 CHICHI SUAREZ MERIDA YUC MX GMY 21101...,$173.00
7,25/12,g APPLE.COM/BILL 866-712-77,$49.25
8,27/12,l b DLOCAL*DIDI FOOD MX CIUDAD DE MEX MX DME 1...,$143.96
9,28/12,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,$28.80


In [18]:
#if statement_type == 'debit':
#    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_reconstructed_table.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_reconstructed_table.csv')
#    
#reconstructed_table.to_csv(reconstructed_table_path, index=False)

In [19]:
from models import DataProcessingFacade

data_processor = DataProcessingFacade(corrected_extracted_words, reconstructed_table, statement_properties)
df_transactions = data_processor.get_normalized_table()
df_transactions

,Date,Description,Amount,Type
0,2024-12-11,y d GASOL CHICHI 2 MERIDA YUC MX CYU 941027P30,-700.00,Cargo
1,2024-12-15,y a CHEDRAUI MERIDA VI MERIDA YUC MX TCH 85070...,-151.10,Cargo
2,2024-12-16,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,-153.46,Cargo
3,2024-12-16,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,-86.78,Cargo
4,2024-12-21,"i SU PAGO, GRACIAS 1/6 Banco Mercantil del ...",1865.00,Abono
5,2024-12-23,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,-79.00,Cargo
6,2024-12-24,y a GM 8 CHICHI SUAREZ MERIDA YUC MX GMY 21101...,-173.00,Cargo
7,2024-12-25,g APPLE.COM/BILL 866-712-77,-49.25,Cargo
8,2024-12-27,l b DLOCAL*DIDI FOOD MX CIUDAD DE MEX MX DME 1...,-143.96,Cargo
9,2024-12-28,l f DLOCAL*DIDI RIDES MX CIUDAD DE MEX MX DME ...,-28.80,Cargo


In [20]:
#from models import CsvExporter
#
#exporter = CsvExporter(df_transactions)
#if statement_type == 'debit':
#    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_normalized_table.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_normalized_table.csv')
#
#exporter.export_data(normalized_table_path)